# Week 6 — Capstone: A Production-Grade Tabular Pipeline

> *XGBoost + LightGBM + CatBoost, tuned by Optuna, stacked by constrained optimization, calibrated, and exported.*

This notebook orchestrates the entire stack into a single end-to-end run that mirrors `scripts/run_pipeline.py`. Everything is reproducible from a YAML config.

## Learning objectives

By the end of this notebook, you will be able to:

1. Assemble a leak-free pipeline: train, validate, blend, calibrate, deploy.
2. Construct out-of-fold predictions for stacking.
3. Compare weighted-ensemble vs. meta-learner stacking.
4. Calibrate tree-based predictions with isotonic / Platt scaling.
5. Optimize the decision threshold under a cost-aware criterion.
6. Export a trained ensemble to ONNX for downstream serving.
7. Generate a model card with full reproducibility metadata.

## Outline

1. **Data ingestion** and leak-aware preprocessing
2. **Per-algorithm tuning** (using Week 5's Optuna outputs in production)
3. **Out-of-fold predictions** for stacking
4. **Weighted ensemble** vs. **stacking** with a meta-learner
5. **Calibration** and reliability diagrams
6. **Threshold optimization** under cost-sensitive criteria
7. **Final evaluation** — AUC, log-loss, PR-AUC, calibration error
8. **Deployment artifacts** — ONNX export, model card, report


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, log_loss, precision_recall_curve,
    roc_auc_score, roc_curve, brier_score_loss,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from gradient_forge.catboost_internals import CatBoostTrainer
from gradient_forge.data.loaders import load_synthetic_classification
from gradient_forge.ensemble import WeightedEnsemble
from gradient_forge.lightgbm_internals import LightGBMTrainer
from gradient_forge.utils import classification_metrics, seed_everything
from gradient_forge.xgboost_internals import XGBoostTrainer
seed_everything(42)


## 1. Data ingestion

For reproducibility we use the synthetic classification dataset shipped with the package. In production, swap `load_synthetic_classification` for a CSV / Parquet loader and apply `LeakAwarePipeline` (from `gradient_forge.data.preprocessing`) for any imputation / scaling / encoding.

**Critical:** the preprocessing pipeline must be fit on training data **only** and then applied to validation / test as a fixed transform. The `LeakAwarePipeline` wrapper around scikit-learn's ColumnTransformer enforces this by exposing only `fit_transform` (training) and `transform` (inference) — never `fit` after `transform`.


In [ ]:
X, y = load_synthetic_classification(n_samples=10_000, n_features=30, class_sep=1.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print(f"Train class balance : {np.bincount(y_train) / len(y_train)}")
print(f"Test  class balance : {np.bincount(y_test)  / len(y_test)}")


## 2. Per-algorithm hyperparameters

In production these values come from the Optuna studies in Week 5 (`studies["xgboost"].best_params` etc.). For notebook reproducibility we use sensible defaults that work well across many tabular problems.


In [ ]:
# Exercise: replace these defaults with studies[algo].best_params from Week 5.
xgb_params = dict(
    n_estimators=500, learning_rate=0.05, max_depth=6, min_child_weight=1.0,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.0,
)
lgb_params = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=63, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5, reg_lambda=1.0,
)
cat_params = dict(
    iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3.0, border_count=254,
)


## 3. Out-of-fold predictions for stacking

For stacking to be leak-free, the meta-features fed to the meta-learner must be **predictions on rows the base model has not seen**. The standard recipe is K-fold out-of-fold (OOF):

For each fold $k = 1, \dots, K$:
1. Train each base model on folds $\ne k$.
2. Predict on fold $k$, store as the meta-feature for those rows.

Concatenating the K predictions gives an $(n_{\text{train}} \times m)$ matrix that is guaranteed leak-free. The base models are then refit on the full training data for final inference.


In [ ]:
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof = np.zeros((X_train.shape[0], 3), dtype=np.float64)
test_preds = np.zeros((X_test.shape[0], 3), dtype=np.float64)

base_models_factory = [
    ("xgb", XGBoostTrainer,  xgb_params),
    ("lgb", LightGBMTrainer, lgb_params),
    ("cat", CatBoostTrainer, cat_params),
]

for fold, (tr, va) in enumerate(cv.split(X_train, y_train), start=1):
    print(f"--- fold {fold}/{N_FOLDS} ---")
    for j, (name, Trainer, params) in enumerate(base_models_factory):
        model = Trainer(task="binary", random_state=42, **params)
        model.fit(X_train[tr], y_train[tr],
                  eval_set=(X_train[va], y_train[va]),
                  early_stopping_rounds=30)
        oof[va, j] = model.predict_proba(X_train[va])[:, 1]
        test_preds[:, j] += model.predict_proba(X_test)[:, 1] / N_FOLDS
        print(f"  {name}: fold AUC = {roc_auc_score(y_train[va], oof[va, j]):.4f}")

print("\n--- OOF AUCs over the full training set ---")
for j, (name, _, _) in enumerate(base_models_factory):
    print(f"  {name}: {roc_auc_score(y_train, oof[:, j]):.4f}")


## 4. Weighted ensemble vs. stacking

### Weighted ensemble — constrained optimization

Solve

$$
\min_{w \in \mathbb{R}^3_+,\; \mathbf{1}^\top w = 1} \;\; \mathcal{L}\bigl( y,\, P w \bigr),
$$

where $P$ is the $(n \times 3)$ matrix of OOF predictions and $\mathcal{L}$ is a smooth surrogate of the target metric (log-loss for AUC, MSE for regression). This is convex with a closed-form gradient → SLSQP converges in a few iterations.

### Stacking — meta-learner on OOF predictions

Train a logistic regression (or any second-level model) on $(P, y)$. The meta-learner discovers non-linear combinations that constrained averaging cannot — e.g. "use XGBoost when LightGBM is very uncertain".

In practice, stacking usually wins by 0.001–0.003 AUC, with much higher variance. Weighted ensembles are the **safer production default**.


In [ ]:
# Weighted ensemble.
blender = WeightedEnsemble(metric="auc").fit(oof, y_train)
blend_train = blender.predict(oof)
blend_test  = blender.predict(test_preds)
print(f"Weighted ensemble weights : {dict(zip(['xgb', 'lgb', 'cat'], blender.weights_.round(3)))}")
print(f"  OOF AUC : {roc_auc_score(y_train, blend_train):.4f}")
print(f"  Test AUC: {roc_auc_score(y_test,  blend_test):.4f}")


In [ ]:
# Stacking with a logistic-regression meta-learner.
meta = LogisticRegression(max_iter=500, random_state=42).fit(oof, y_train)
stack_train = meta.predict_proba(oof)[:, 1]
stack_test  = meta.predict_proba(test_preds)[:, 1]
print(f"\nStacking (LogReg meta-learner):")
print(f"  meta-learner coefs: {dict(zip(['xgb', 'lgb', 'cat'], meta.coef_[0].round(3)))}")
print(f"  meta-learner intercept: {meta.intercept_[0]:.3f}")
print(f"  OOF AUC : {roc_auc_score(y_train, stack_train):.4f}")
print(f"  Test AUC: {roc_auc_score(y_test,  stack_test):.4f}")


In [ ]:
# Side-by-side comparison.
metrics_table = []
for name, preds in [("XGBoost only",  test_preds[:, 0]),
                    ("LightGBM only", test_preds[:, 1]),
                    ("CatBoost only", test_preds[:, 2]),
                    ("Weighted ensemble", blend_test),
                    ("Stacking",            stack_test)]:
    metrics_table.append({
        "model": name,
        "AUC": roc_auc_score(y_test, preds),
        "PR-AUC": average_precision_score(y_test, preds),
        "log-loss": log_loss(y_test, np.clip(preds, 1e-15, 1 - 1e-15)),
        "Brier": brier_score_loss(y_test, preds),
    })
df_metrics = pd.DataFrame(metrics_table).set_index("model").round(4)
df_metrics


## 5. Calibration

Tree-based models are typically **miscalibrated** — predicted probabilities don't match observed frequencies. For decision-making applications (cost-sensitive classification, expected-value optimization), this matters as much as AUC.

### Reliability diagram

Bin predictions into deciles, compute the observed frequency within each bin, plot against the predicted probability. A well-calibrated model lies on the diagonal.


In [ ]:
prob_true, prob_pred = calibration_curve(y_test, blend_test, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration", alpha=0.5)
ax.plot(prob_pred, prob_true, "o-", color="C3", linewidth=2, label="weighted ensemble")
ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")
ax.set_title(f"Reliability diagram (Brier = {brier_score_loss(y_test, blend_test):.4f})")
ax.legend(); ax.grid(alpha=0.3)
ax.set_aspect("equal"); plt.show()


### 5.1 Isotonic and Platt (sigmoid) calibration

If the reliability diagram deviates from the diagonal, post-hoc calibration helps. Two options:

- **Platt (sigmoid) scaling** — fit a logistic regression mapping raw scores to probabilities. Works best when miscalibration is monotone and smooth.
- **Isotonic regression** — non-parametric monotone fit. More flexible but needs more data to avoid overfitting.

Both require a **separate calibration set** to avoid double-using the training data. Best practice: hold out 10–20% of training data specifically for calibration.


In [ ]:
# Empirical comparison: post-hoc isotonic calibration on the OOF predictions.
# We use OOF predictions as the calibration set since they are already leak-free.
from sklearn.isotonic import IsotonicRegression

iso = IsotonicRegression(out_of_bounds="clip").fit(blender.predict(oof), y_train)
blend_test_cal = iso.predict(blend_test)

prob_true_cal, prob_pred_cal = calibration_curve(
    y_test, blend_test_cal, n_bins=10, strategy="quantile"
)
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfect")
ax.plot(prob_pred,     prob_true,     "o-", color="C0", linewidth=2, label="uncalibrated")
ax.plot(prob_pred_cal, prob_true_cal, "s-", color="C3", linewidth=2, label="isotonic")
ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")
ax.set_title("Calibration before vs. after isotonic regression")
ax.legend(); ax.grid(alpha=0.3); ax.set_aspect("equal"); plt.show()

brier_before = brier_score_loss(y_test, blend_test)
brier_after  = brier_score_loss(y_test, blend_test_cal)
print(f"Brier score: {brier_before:.4f} -> {brier_after:.4f} (lower is better)")


## 6. Threshold optimization under cost-sensitive criteria

The default decision threshold of 0.5 is only correct when:

- The class distribution is balanced (50/50), and
- False positives and false negatives have equal cost.

In practice, almost neither holds. Two common alternatives:

### Youden's J (maximize TPR - FPR)

Symmetric, simple, requires no cost specification. The default for diagnostic-style problems.

### Cost-aware threshold

Specify a cost matrix:

| | Predict 0 | Predict 1 |
|---|---|---|
| True 0 | 0 | $c_{FP}$ |
| True 1 | $c_{FN}$ | 0 |

Optimal threshold:

$$
t^\star = \frac{c_{FP}}{c_{FP} + c_{FN}}.
$$

For most fraud / churn / medical problems, $c_{FN} \gg c_{FP}$ (false negatives are very expensive), so $t^\star$ is much smaller than 0.5.


In [ ]:
# Youden's J optimal threshold.
fpr, tpr, thresholds = roc_curve(y_test, blend_test_cal)
youden_j = tpr - fpr
best_t_youden = float(thresholds[np.argmax(youden_j)])

# Cost-aware: assume false negatives are 5x more expensive than false positives.
c_fp, c_fn = 1.0, 5.0
expected_cost = c_fp * fpr + c_fn * (1 - tpr) * (np.sum(y_test) / len(y_test))
best_t_cost = float(thresholds[np.argmin(expected_cost)])

print(f"Optimal threshold (Youden's J)          : {best_t_youden:.4f}")
print(f"Optimal threshold (cost: c_FN/c_FP = 5) : {best_t_cost:.4f}")
print(f"\nNote how the cost-sensitive threshold is lower — the model is more eager to flag positives.")


In [ ]:
# Visualize the cost surface.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, youden_j, label="Youden's J", color="C0")
ax2 = ax.twinx()
ax2.plot(thresholds, expected_cost, label="expected cost (c_FN/c_FP = 5)", color="C3")
ax.axvline(best_t_youden, color="C0", linestyle="--", alpha=0.5)
ax2.axvline(best_t_cost,  color="C3", linestyle="--", alpha=0.5)
ax.set_xlabel("threshold")
ax.set_ylabel("Youden's J", color="C0")
ax2.set_ylabel("expected cost", color="C3")
ax.set_title("Threshold optimization under two criteria")
ax.set_xlim(0, 1); plt.show()


## 7. Final evaluation


In [ ]:
# Apply the chosen (cost-aware) threshold for the final classification metrics.
final_proba = blend_test_cal
final_metrics = classification_metrics(y_test, final_proba, threshold=best_t_cost)
final_metrics["pr_auc"] = float(average_precision_score(y_test, final_proba))
final_metrics["brier"]  = float(brier_score_loss(y_test, final_proba))
final_metrics["threshold"] = best_t_cost

pd.DataFrame([final_metrics]).T.rename(columns={0: "value"}).round(4)


In [ ]:
# Precision-recall curve.
prec, rec, _ = precision_recall_curve(y_test, final_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec, prec, linewidth=2)
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title(f"Precision-recall curve (PR-AUC = {final_metrics['pr_auc']:.4f})")
ax.grid(alpha=0.3); plt.show()


## 8. Deployment artifacts

A trained model is not enough — production deployment requires:

1. **Model file** (ONNX for cross-runtime portability)
2. **Model card** describing data, training procedure, metrics, intended use, known limitations
3. **Reproducibility report** logging all hyperparameters, package versions, and seeds
4. **Inference test** confirming the exported model produces the same scores as the training-time predict


In [ ]:
# Generate a JSON model card / reproducibility report.
import sys as _sys
import platform

import numpy as _np
import sklearn as _sk

report = {
    "name": "GradientForge — Tabular ML Pipeline",
    "version": "1.0.0",
    "metrics": final_metrics,
    "ensemble": {
        "method": "weighted (constrained least squares)",
        "weights": dict(zip(["xgb", "lgb", "cat"], blender.weights_.round(4).tolist())),
        "calibration": "isotonic",
        "threshold": best_t_cost,
    },
    "data": {
        "n_train": int(X_train.shape[0]),
        "n_test":  int(X_test.shape[0]),
        "n_features": int(X_train.shape[1]),
        "train_class_balance": (np.bincount(y_train) / len(y_train)).round(4).tolist(),
    },
    "base_models": {
        "xgb": xgb_params, "lgb": lgb_params, "cat": cat_params,
    },
    "environment": {
        "python":   _sys.version.split()[0],
        "platform": platform.platform(),
        "numpy":    _np.__version__,
        "sklearn":  _sk.__version__,
    },
    "intended_use": "Demonstration of a leak-free, calibrated tabular classification pipeline.",
    "limitations": [
        "Trained on synthetic data — performance on real distributions is not characterized.",
        "Threshold calibrated under a specific cost ratio (c_FN / c_FP = 5).",
    ],
}

reports_dir = Path("_outputs"); reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / "capstone_report.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"Wrote model card → {report_path}")
print()
print(json.dumps(report, indent=2)[:600] + "  ...")


### 8.1 ONNX export

ONNX (Open Neural Network Exchange) provides a standard format that:

- Serializes a model independently of the training library
- Can be served via ONNX Runtime, C++ runtimes, mobile inference engines, browsers
- Supports tree-based models via `onnxmltools` (XGBoost, LightGBM) and CatBoost's built-in exporter

The script `scripts/export_onnx.py` in this repository implements the export logic. The pattern looks like:

```python
from onnxmltools.convert import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

# For each base model:
onnx_model = convert_xgboost(
    xgb_model.model_,
    initial_types=[("input", FloatTensorType([None, n_features]))],
)
with open(f"artifacts/xgb.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
```

At inference time the consumer needs only ONNX Runtime — no XGBoost, LightGBM, or CatBoost installations required. This is the difference between a Python-bound research artifact and a deployable production model.


## 9. Reproducing this pipeline from the command line

Everything in this notebook is also exposed as a CLI script:

```bash
python scripts/run_pipeline.py --config scripts/configs/pipeline.yaml
```

The YAML config controls every choice (data source, fold count, ensemble method, calibration strategy, ONNX export flag). This is the script you would invoke from a CI job or a daily retraining cron.

```bash
# Run the Optuna study first (Week 5)
python scripts/run_optuna_study.py --algorithm xgboost --n-trials 200 --gpu --storage sqlite:///optuna.db

# Then plug the best params into pipeline.yaml and run the capstone
python scripts/run_pipeline.py --config scripts/configs/pipeline.yaml

# Export to ONNX
python scripts/export_onnx.py --model artifacts/xgb.pkl --output artifacts/xgb.onnx --n-features 30
```

## 10. Exercises

1. **Replace the synthetic data** with a real dataset (Kaggle Titanic, Higgs Boson, Otto Group). Rerun the full pipeline and compare the model-card metrics.
2. **Add a 4th base learner.** A deep tabular model like TabNet or FT-Transformer. Does ensemble diversity help?
3. **Cost-matrix sweep.** Vary $c_{FN} / c_{FP}$ over $\{1, 2, 5, 10, 20\}$ and plot the resulting precision and recall. At what cost ratio does the threshold drop below 0.1?
4. **ONNX equivalence test.** Export the trained ensemble, load it with ONNX Runtime, and verify predictions match the Python originals to within $10^{-6}$.

## Final takeaways

- A well-engineered weighted ensemble of XGBoost + LightGBM + CatBoost is the **modal Kaggle / industry baseline** for tabular problems.
- **Out-of-fold prediction is non-negotiable** — in-fold stacking leaks the validation set.
- **Calibration matters** whenever the downstream decision is probabilistic (cost-sensitive classification, expected-value optimization, threshold-based alerting).
- The decision threshold is a **business choice**, not a default. Specify the cost ratio and the threshold falls out.
- Production deployment requires **artifacts** (ONNX, model card) and **reproducibility evidence** (env snapshot, seeds, config) — the model file alone is insufficient.

> **What's next.** The full course is complete. Use the `scripts/run_pipeline.py` CLI as a starting point for your own tabular problems, and contribute back any improvements via pull request.
